In [1]:
import numpy as np
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(0)

def ece(conf, correct, mode="uniform", n_bins=15):
    """conf: 置信度; correct: 0/1; mode: uniform(等宽) / adaptive(等样本量)"""
    if mode == "uniform":
        edges = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(conf, edges[1:-1]), 0, n_bins - 1)
    else:                                   # 等样本量分箱
        order = np.argsort(conf)
        idx = np.empty(len(conf), dtype=int)
        idx[order] = np.floor(np.arange(len(conf)) * n_bins / len(conf)).astype(int)
        idx = np.clip(idx, 0, n_bins - 1)
    e = 0.0
    for b in np.unique(idx):
        m = idx == b
        if m.sum() == 0:
            continue
        e += m.sum() / len(conf) * abs(correct[m].mean() - conf[m].mean())
    return e

def ece_bootstrap(conf, correct, B=500, **kw):
    n = len(conf)
    vals = [ece(conf[i], correct[i], **kw)
            for i in (rng.integers(0, n, (B, n)))]
    return np.percentile(vals, [2.5, 97.5])

def fit_temperature(logits, y):
    def nll(T):
        z = logits / T
        z = z - z.max(axis=1, keepdims=True)
        return -(z[np.arange(len(y)), y] - np.log(np.exp(z).sum(1))).mean()
    return minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x

# 模拟一个过度自信的模型: 真实正确率 0.75, 自报置信度均值 0.90
n = 4000
correct = rng.random(n) < 0.75
conf = np.clip(rng.beta(9, 1.4, n) * (0.55 + 0.45 * correct), 1e-6, 1 - 1e-6)

print(f"ECE 等宽分箱   = {ece(conf, correct, 'uniform'):.4f}"
      f"   95% CI {np.round(ece_bootstrap(conf, correct, mode='uniform'), 4)}")
print(f"ECE 等样本分箱 = {ece(conf, correct, 'adaptive'):.4f}"
      f"   95% CI {np.round(ece_bootstrap(conf, correct, mode='adaptive'), 4)}")
for nb in [5, 10, 15, 30, 50]:
    print(f"  分箱数 M={nb:2d}: ECE={ece(conf, correct, 'uniform', nb):.4f}"
          f"   (完全校准模型也会因噪声而 > 0)")

# 温度缩放: 由置信度反推 logits 的简化演示(二分类)
p = np.clip(conf, 1e-6, 1 - 1e-6)
logits = np.column_stack([np.log(1 - p), np.log(p)])
T = fit_temperature(logits, correct.astype(int))
conf_cal = 1 / (1 + np.exp(-(logits[:, 1] - logits[:, 0]) / T))
print(f"拟合温度 T = {T:.3f}  (>1 说明原模型过度自信)")
print(f"标定后 ECE = {ece(conf_cal, correct, 'uniform'):.4f}")

ECE 等宽分箱   = 0.2089   95% CI [0.2024 0.2142]
ECE 等样本分箱 = 0.2071   95% CI [0.1915 0.2197]
  分箱数 M= 5: ECE=0.2089   (完全校准模型也会因噪声而 > 0)
  分箱数 M=10: ECE=0.2089   (完全校准模型也会因噪声而 > 0)
  分箱数 M=15: ECE=0.2089   (完全校准模型也会因噪声而 > 0)
  分箱数 M=30: ECE=0.2143   (完全校准模型也会因噪声而 > 0)
  分箱数 M=50: ECE=0.2156   (完全校准模型也会因噪声而 > 0)
拟合温度 T = 0.179  (>1 说明原模型过度自信)
标定后 ECE = 0.1015


In [2]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(1)

def conformal_quantile(scores, alpha=0.1):
    """保形阈值: 第 ceil((n+1)(1-alpha)) 小的标定分数"""
    s = np.sort(scores)
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))          # 1-indexed 的第 k 小
    return s[min(k, n) - 1]

def split_conformal(X_tr, y_tr, X_cal, y_cal, X_test, alpha=0.1, model=None):
    """回归的分割保形: 分数 = |y - f(x)|"""
    model = model or GradientBoostingRegressor(random_state=0)
    model.fit(X_tr, y_tr)
    s_cal = np.abs(y_cal - model.predict(X_cal))
    q = conformal_quantile(s_cal, alpha)
    pred = model.predict(X_test)
    return pred - q, pred + q, q

def conformal_qr(X_tr, y_tr, X_cal, y_cal, X_test, alpha=0.1):
    """CQR: 分数由条件分位数残差构造, 区间宽度随 x 变化"""
    lo_m = GradientBoostingRegressor(loss="quantile", alpha=alpha / 2, random_state=0)
    hi_m = GradientBoostingRegressor(loss="quantile", alpha=1 - alpha / 2, random_state=0)
    lo_m.fit(X_tr, y_tr); hi_m.fit(X_tr, y_tr)
    lo_c, hi_c = lo_m.predict(X_cal), hi_m.predict(X_cal)
    s = np.maximum(lo_c - y_cal, y_cal - hi_c)
    q = conformal_quantile(s, alpha)
    return lo_m.predict(X_test) - q, hi_m.predict(X_test) + q, q

# 异方差数据: 噪声标准差随 x 增大 -> 检验条件覆盖与区间宽度
n_all = 4000
X = rng.uniform(-3, 3, (n_all, 3))
y = np.sin(X[:, 0]) + 0.5 * X[:, 1] ** 2 + (0.2 + 0.6 * np.abs(X[:, 2])) * rng.normal(0, 1, n_all)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.5, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=1)

for name, fn in [("split conformal", split_conformal), ("CQR", conformal_qr)]:
    lo, hi, q = fn(X_tr, y_tr, X_cal, y_cal, X_te, alpha=0.1)
    cov = np.mean((y_te >= lo) & (y_te <= hi))
    width = np.mean(hi - lo)
    # 条件覆盖: 按 |x2| 分层检查
    g = np.abs(X_te[:, 2]) > np.median(np.abs(X_te[:, 2]))
    cov_lo = np.mean((y_te[g] >= lo[g]) & (y_te[g] <= hi[g]))
    cov_hi = np.mean((y_te[~g] >= lo[~g]) & (y_te[~g] <= hi[~g]))
    print(f"{name:16s} q={q:.3f} 边际覆盖={cov:.3f} 平均宽度={width:.3f} "
          f"| 子群覆盖: 高噪={cov_lo:.3f} 低噪={cov_hi:.3f}")
# 边际覆盖接近 0.90, 但子群之间可能偏离: 这是边际覆盖与条件覆盖的差距

split conformal  q=2.069 边际覆盖=0.895 平均宽度=4.138 | 子群覆盖: 高噪=0.802 低噪=0.988
CQR              q=0.075 边际覆盖=0.880 平均宽度=3.830 | 子群覆盖: 高噪=0.850 低噪=0.910


In [3]:
import numpy as np
from scipy.stats import norm
rng = np.random.default_rng(2)

def wilson_ci(k, n, alpha=0.05):
    """二项比例的 Wilson 区间: 小样本与极端比例下覆盖优于 Wald"""
    z = norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return center - half, center + half

def cluster_bootstrap(per_item_correct, n_item, B=2000):
    """按问题聚类的 bootstrap: 每个问题是一个 cluster"""
    vals = []
    for _ in range(B):
        idx = rng.integers(0, n_item, n_item)
        vals.append(per_item_correct[idx].mean())
    return np.percentile(vals, [2.5, 97.5])

# 模拟: 800 个问题, 每题 5 个输出, 类内相关 -> 幻觉率约 0.22
n_item, m = 800, 5
p_hallu = 0.22
item_hallu = (rng.random(n_item) < p_hallu)          # 问题层: 该题是否会触发幻觉
out_hallu = np.where(item_hallu[:, None],
                     rng.random((n_item, m)) < 0.7,  # 若会, 约 70% 的输出出错
                     rng.random((n_item, m)) < 0.05) # 若不会, 偶发错误

flat = out_hallu.ravel()
print(f"朴素(把输出当独立样本): p={flat.mean():.3f}  "
      f"Wald CI={flat.mean():.3f}±{1.96*np.sqrt(flat.mean()*(1-flat.mean())/flat.size):.3f}")
k = out_hallu.sum(); n_out = out_hallu.size
print(f"Wilson 区间(仍忽略聚类) = {np.round(wilson_ci(k, n_out), 3)}")
print(f"聚类 bootstrap(按问题)   = {np.round(cluster_bootstrap(out_hallu.mean(axis=1), n_item), 3)}")
# 聚类区间更宽: 忽略聚类会低估不确定性

def n_required(delta=0.03, p=0.5, alpha=0.05, m=5, rho=0.3):
    z = norm.ppf(1 - alpha / 2)
    deff = 1 + (m - 1) * rho                          # 设计效应
    return int(np.ceil(z**2 * p * (1 - p) / delta**2 * deff))
print(f"目标半宽 0.03, 每题 5 输出, rho=0.3 -> 需要问题数 ≈ {n_required()}")

朴素(把输出当独立样本): p=0.196  Wald CI=0.196±0.012
Wilson 区间(仍忽略聚类) = [0.184 0.209]
聚类 bootstrap(按问题)   = [0.175 0.217]
目标半宽 0.03, 每题 5 输出, rho=0.3 -> 需要问题数 ≈ 2348


In [ ]:
# import numpy as np
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score
# rng = np.random.default_rng(3)

# def median_heuristic(X, Y):
#     """RBF 带宽: 取样本间距离的中位数(标准启发式)"""
#     Z = np.vstack([X[:300], Y[:300]])
#     D = np.sqrt(((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1))
#     return float(np.median(D[D > 0])) + 1e-8

# def rbf_kernel(X, Y, sigma=None):
#     if sigma is None:
#         sigma = median_heuristic(X, Y)
#     X2 = (X**2).sum(1)[:, None]; Y2 = (Y**2).sum(1)[None, :]
#     K = np.exp(-(X2 + Y2 - 2 * X @ Y.T) / (2 * sigma**2))
#     return K

# def mmd2(X, Y, sigma=None):
#     Kxx = rbf_kernel(X, X, sigma).mean()
#     Kyy = rbf_kernel(Y, Y, sigma).mean()
#     Kxy = rbf_kernel(X, Y, sigma).mean()
#     return Kxx + Kyy - 2 * Kxy

# def mmd_permutation_test(X, Y, n_perm=300, sigma=None):
#     stat = mmd2(X, Y, sigma)
#     Z = np.vstack([X, Y]); n = len(X)
#     null = []
#     for _ in range(n_perm):
#         perm = rng.permutation(len(Z))
#         null.append(mmd2(Z[perm[:n]], Z[perm[n:]], sigma))
#     return stat, (1 + np.sum(np.array(null) >= stat)) / (1 + n_perm)

# def c2st(X, Y):
#     Xs, Ys = np.vstack([X, np.zeros((len(X), 1))]), np.vstack([Y, np.ones((len(Y), 1))])
#     D = np.vstack([Xs, Ys]); lab = np.r_[np.zeros(len(X)), np.ones(len(Y))]
#     Dtr, Dte, ltr, lte = train_test_split(D, lab, test_size=0.4, random_state=0)
#     clf = LogisticRegression(max_iter=1000).fit(Dtr, ltr)
#     return roc_auc_score(lte, clf.predict_proba(Dte)[:, 1])

# # 真实数据: 两成分混合; 合成数据: 只覆盖其中一个成分(模式坍塌)
# X = np.vstack([rng.normal(-2, 1, (400, 5)), rng.normal(2, 1, (400, 5))])
# Y_good = np.vstack([rng.normal(-2, 1, (400, 5)), rng.normal(2, 1, (400, 5))])
# Y_bad = rng.normal(-2, 1, (800, 5))                     # 只学到了一个模式

# for name, Ys in [("保真且多样", Y_good), ("模式坍塌", Y_bad)]:
#     stat, p = mmd_permutation_test(X, Ys, n_perm=200)
#     print(f"{name}: MMD^2={stat:.5f}  置换 p={p:.3f}  C2ST AUC={c2st(X, Ys):.3f}")
# # 模式坍塌的合成数据在两个指标上都应被检出; 0.5 附近表示分布接近

In [5]:
import numpy as np

rng = np.random.default_rng(17)

def run_once(n, delta, rho, method):
    """单次实验: T 50/50 随机化, X 为前协变量(处理前测量), Y 为结果"""
    T = rng.integers(0, 2, n)
    X = rng.normal(0, 1, n)
    # 参数化保证 corr(X, Y) = rho: Var(Y|T) = 1
    Y = delta * T + rho * X + np.sqrt(1 - rho**2) * rng.normal(0, 1, n)
    if method == "naive":
        est = Y[T == 1].mean() - Y[T == 0].mean()
    else:  # CUPED: theta = Cov(X, Y) / Var(X), 调整量 theta*(X - mean(X))
        theta = np.cov(X, Y)[0, 1] / X.var()
        Yc = Y - theta * (X - X.mean())      # X 是处理前测量, 调整不影响无偏性
        est = Yc[T == 1].mean() - Yc[T == 0].mean()
    return est

n, delta, rho, R = 200, 0.30, 0.70, 20000
ests = {m: np.array([run_once(n, delta, rho, m) for _ in range(R)])
        for m in ["naive", "cuped"]}

sd_ratio = ests["cuped"].std() / ests["naive"].std()
print(f"前协变量与结果的相关 rho = {rho}")
print(f"经验标准误: 朴素 = {ests['naive'].std():.4f}  CUPED = {ests['cuped'].std():.4f}")
print(f"标准误之比 = {sd_ratio:.3f}   理论值 sqrt(1-rho^2) = {np.sqrt(1-rho**2):.3f}")
print(f"均值: 朴素 = {ests['naive'].mean():+.4f}  CUPED = {ests['cuped'].mean():+.4f}"
      f"   (真效应 delta={delta}, 两者均无偏)")
print(f"等效样本量放大 = 1/(1-rho^2) = {1/(1-rho**2):.2f} 倍")

前协变量与结果的相关 rho = 0.7
经验标准误: 朴素 = 0.1424  CUPED = 0.1010
标准误之比 = 0.709   理论值 sqrt(1-rho^2) = 0.714
均值: 朴素 = +0.2995  CUPED = +0.2992   (真效应 delta=0.3, 两者均无偏)
等效样本量放大 = 1/(1-rho^2) = 1.96 倍


In [6]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(4)

n_items, n_seed = 300, 5
# 题目难度 latent, 两模型共享 -> 配对设计可消除题目主效应
difficulty = rng.normal(0, 1.2, n_items)
def simulate(p_base, rho=0.85):
    shared = np.sqrt(rho) * difficulty + np.sqrt(1 - rho) * rng.normal(0, 1, n_items)
    return (shared > -stats.norm.ppf(p_base)).astype(int)
A = simulate(0.62); B = simulate(0.66)          # B 略好

a = int(((A == 1) & (B == 1)).sum()); b = int(((A == 1) & (B == 0)).sum())
c = int(((A == 0) & (B == 1)).sum()); d = int(((A == 0) & (B == 0)).sum())
chi2 = (abs(b - c) - 1) ** 2 / (b + c)
p_chi = 1 - stats.chi2.cdf(chi2, df=1)
p_exact = stats.binomtest(b, b + c, 0.5).pvalue
print(f"四格表 a={a} b={b} c={c} d={d}")
print(f"McNemar chi2={chi2:.3f}  p={p_chi:.4f}   精确检验 p={p_exact:.4f}")
print(f"边际准确率 A={A.mean():.3f} B={B.mean():.3f}  差值={B.mean()-A.mean():+.3f}")

# cluster bootstrap: 以题目为聚类单位重抽样
def cluster_boot_ci(diff, B=5000):
    n = len(diff)
    means = diff[rng.integers(0, n, (B, n))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])
print(f"配对差值 95% CI (cluster bootstrap) = {np.round(cluster_boot_ci(B - A), 4)}")

# 两比例 z 检验(忽略配对) 与配对检验的功效对比
sd_pair = (B - A).std(ddof=1)
sd_unpair = np.sqrt(A.mean()*(1-A.mean()) + B.mean()*(1-B.mean()))
print(f"配对 SE={sd_pair/np.sqrt(n_items):.4f}  非配对 SE={sd_unpair/np.sqrt(n_items):.4f}"
      f"  效率比={(sd_unpair/sd_pair)**2:.2f}x")

# 功效分析: 检出 delta 需要多少题目
for delta in [0.02, 0.05, 0.10]:
    n_req = (stats.norm.ppf(0.975) + stats.norm.ppf(0.8))**2 * sd_pair**2 / delta**2
    print(f"MDE={delta:.2f} -> 需要题目数 n≈{int(np.ceil(n_req))} (功效 80%)")

四格表 a=173 b=20 c=19 d=88
McNemar chi2=0.000  p=1.0000   精确检验 p=1.0000
边际准确率 A=0.643 B=0.640  差值=-0.003
配对差值 95% CI (cluster bootstrap) = [-0.0433  0.0367]
配对 SE=0.0209  非配对 SE=0.0392  效率比=3.53x
MDE=0.02 -> 需要题目数 n≈2560 (功效 80%)
MDE=0.05 -> 需要题目数 n≈410 (功效 80%)
MDE=0.10 -> 需要题目数 n≈103 (功效 80%)


In [7]:
import numpy as np
rng = np.random.default_rng(7)

d, k = 512, 512
# 模拟一个谱快速衰减的预训练权重(能量集中在前几个奇异方向)
U = np.linalg.qr(rng.normal(0, 1, (d, d)))[0]
V = np.linalg.qr(rng.normal(0, 1, (k, k)))[0]
s = 10.0 * np.exp(-np.arange(min(d, k)) / 12)      # 指数衰减的奇异值
W0 = U[:, :len(s)] @ np.diag(s) @ V[:, :len(s)].T

sv = np.linalg.svd(W0, compute_uv=False)
stable_rank = (sv ** 2).sum() / sv[0] ** 2
print(f"稳定秩 (||W||_F^2 / ||W||_2^2) = {stable_rank:.1f}   (远小于 min(d,k)=512)")

for r in [1, 2, 4, 8, 16, 32, 64]:
    Ur, sr, Vtr = np.linalg.svd(W0, full_matrices=False)
    W_r = (Ur[:, :r] * sr[:r]) @ Vtr[:r]
    rel = np.linalg.norm(W0 - W_r) / np.linalg.norm(W0)
    energy = (sr[:r] ** 2).sum() / (sr ** 2).sum()
    params = r * (d + k)
    print(f"  r={r:2d}: 相对近似误差={rel:.4f}  累积能量={energy:.4f}  "
          f"可训练参数={params}  占全量 {params/(d*k)*100:.2f}%")
# 谱衰减快时, 很小的 r 即可捕获大部分能量 -> 低秩假设在该矩阵上成立

稳定秩 (||W||_F^2 / ||W||_2^2) = 6.5   (远小于 min(d,k)=512)
  r= 1: 相对近似误差=0.9200  累积能量=0.1535  可训练参数=1024  占全量 0.39%
  r= 2: 相对近似误差=0.8465  累积能量=0.2835  可训练参数=2048  占全量 0.78%
  r= 4: 相对近似误差=0.7165  累积能量=0.4866  可训练参数=4096  占全量 1.56%
  r= 8: 相对近似误差=0.5134  累积能量=0.7364  可训练参数=8192  占全量 3.12%
  r=16: 相对近似误差=0.2636  累积能量=0.9305  可训练参数=16384  占全量 6.25%
  r=32: 相对近似误差=0.0695  累积能量=0.9952  可训练参数=32768  占全量 12.50%
  r=64: 相对近似误差=0.0048  累积能量=1.0000  可训练参数=65536  占全量 25.00%


In [ ]:
# import numpy as np, pandas as pd
# rng = np.random.default_rng(8)
# n = 5000
# difficulty = rng.normal(0, 1, n)                      # 混杂: 影响是否被检索 + 影响正确率
# retrieved = (rng.random(n) < 0.5 + 0.25 * np.tanh(difficulty))   # 受难度影响(非随机)
# true_effect = 0.12
# correct = (rng.random(n) < 0.55 + true_effect * retrieved - 0.20 * difficulty).astype(int)
# df = pd.DataFrame({"difficulty": difficulty, "retrieved": retrieved.astype(int),
#                    "correct": correct})

# from dowhy import CausalModel
# model = CausalModel(data=df,
#                     treatment="retrieved", outcome="correct",
#                     common_causes=["difficulty"])       # 步骤1: 建模(显式写出混杂)
# identified = model.identify_effect()                    # 步骤2: 识别
# est = model.estimate_effect(identified,
#                             method_name="backdoor.propensity_score_stratification")
# print(f"调整后估计 (ATE) = {est.value:.4f}   朴素差值 = "
#       f"{df[df.retrieved==1].correct.mean() - df[df.retrieved==0].correct.mean():.4f}")

# # 步骤4: 反驳检验
# for test in ["random_common_cause", "placebo_treatment_refuter",
#              "data_subset_refuter"]:
#     r = model.refute_estimate(identified, est, method_name=test)
#     print(f"  {test:28s} -> {r}")

In [9]:
import numpy as np

rng = np.random.default_rng(16)

# 真实过程: 振荡 + 线性趋势 + 噪声 (两个候选模型各自只能捕获其中一部分)
def gen(n, rng):
    x = rng.uniform(-3, 3, n)
    y = 0.9 * np.sin(1.5 * x) + 0.35 * x + rng.normal(0, 0.4, n)
    return x, y

x_tr, y_tr = gen(60, rng)
x_te, y_te = gen(2000, rng)

def fit_predict(x_tr, y_tr, x_te, basis):
    X_tr, X_te = basis(x_tr), basis(x_te)
    beta, *_ = np.linalg.lstsq(X_tr, y_tr, rcond=None)
    return X_te @ beta, beta

# 模型 A: 只含振荡项 (误设: 缺趋势); 模型 B: 只含线性项 (误设: 缺振荡)
basis_A = lambda x: np.column_stack([np.sin(1.5 * x), np.cos(1.5 * x)])
basis_B = lambda x: np.column_stack([x, np.ones_like(x)])

pred_A, beta_A = fit_predict(x_tr, y_tr, x_te, basis_A)
pred_B, beta_B = fit_predict(x_tr, y_tr, x_te, basis_B)

def bic(y, pred, k):
    rss = ((y - pred) ** 2).sum()
    return len(y) * np.log(rss / len(y)) + k * np.log(len(y))

n_par = [basis_A(x_tr).shape[1], basis_B(x_tr).shape[1]]
bics = [bic(y_tr, basis_A(x_tr) @ beta_A, n_par[0]),
        bic(y_tr, basis_B(x_tr) @ beta_B, n_par[1])]
# BIC 权重 = 边际似然的 BIC 近似 softmax
logw = -0.5 * np.array(bics)
w = np.exp(logw - logw.max()); w /= w.sum()
pred_BMA = w[0] * pred_A + w[1] * pred_B

rmse = lambda p: float(np.sqrt(((p - y_te) ** 2).mean()))
# 两个模型预测误差的相关性: 越低, 组合收益越大
eA, eB = y_te - pred_A, y_te - pred_B
corr_err = np.corrcoef(eA, eB)[0, 1]

print(f"BIC: A={bics[0]:.1f}  B={bics[1]:.1f}")
print(f"BIC 权重: w_A={w[0]:.3f}  w_B={w[1]:.3f}   (均不为 0 -> 有限样本下未退化为单模型)")
print(f"测试 RMSE: 模型A={rmse(pred_A):.4f}  模型B={rmse(pred_B):.4f}  BMA组合={rmse(pred_BMA):.4f}")
print(f"两模型预测误差相关 = {corr_err:.3f}   (组合收益来自误差互补, 见 M-open 讨论)")

BIC: A=-20.9  B=-23.5
BIC 权重: w_A=0.213  w_B=0.787   (均不为 0 -> 有限样本下未退化为单模型)
测试 RMSE: 模型A=0.7354  模型B=0.7536  BMA组合=0.6693
两模型预测误差相关 = 0.387   (组合收益来自误差互补, 见 M-open 讨论)


In [13]:



import numpy as np
from scipy.optimize import minimize
rng = np.random.default_rng(9)

# --- (1) 基于模拟的校准 (SBC): 检查推断算法的秩分布是否均匀 ---
def sbc_ranks(n_sim=1000, n_post=200):
    ranks = []
    for _ in range(n_sim):
        theta = rng.normal(0, 1)                       # 从先验抽样
        y_obs = rng.normal(theta, 1, 5)                # 生成数据
        # 共轭正态后验: 先验 N(0,1), 似然 N(theta,1), n=5
        prec = 1 + len(y_obs)
        mean = y_obs.sum() / prec
        post = rng.normal(mean, 1 / np.sqrt(prec), n_post)
        ranks.append(np.sum(post < theta))             # theta 在后验样本中的秩
    return np.array(ranks)

ranks = sbc_ranks()
counts = np.bincount(ranks, minlength=201)[:201]
print("SBC 秩分布均匀性检验 (卡方):")
exp = len(ranks) / 201
chi2 = ((counts - exp) ** 2 / exp).sum()
from scipy.stats import chi2 as chi2dist
print(f"  chi2={chi2:.1f}  df=200  p={1-chi2dist.cdf(chi2, 200):.3f}  "
      f"(p 很小说明推断算法不自洽)")



# --- stacking ---
def stacking_weights(preds, y):
    K = preds.shape[1]
    def neg_obj(u):
        w = np.exp(u - u.max()); w /= w.sum()
        return -np.log(preds @ w + 1e-300).mean()
    res = minimize(neg_obj, np.zeros(K), method="L-BFGS-B")
    w = np.exp(res.x - res.x.max()); w /= w.sum()
    return w

n, K = 800, 3
y = rng.integers(0, 5, n)

# 修复点：y[:, None] vs np.arange(5)[None, :] -> (n, 5)
mask = np.arange(5)[None, :] == y[:, None]
preds = np.column_stack([
    np.where(mask, 0.60, 0.10),
    np.where(mask, 0.40, 0.15),
    np.where(mask, 0.22, 0.195),
])
preds = preds / preds.sum(1, keepdims=True)

w = stacking_weights(preds, y)
print(f"stacking 权重 = {np.round(w, 3)}   (等权 = {np.round([1/3]*3, 3)})")
print(f"堆叠后对数得分 = {np.log(preds @ w).mean():.4f}   "
      f"最优单模型 = {max(np.log(preds[:, k]).mean() for k in range(K)):.4f}")


SBC 秩分布均匀性检验 (卡方):
  chi2=210.4  df=200  p=0.293  (p 很小说明推断算法不自洽)
stacking 权重 = [0.055 0.04  0.04  0.098 0.128 0.059 0.05  0.05  0.078 0.088 0.062 0.061
 0.061 0.064 0.065]   (等权 = [0.333 0.333 0.333])
堆叠后对数得分 = -2.7024   最优单模型 = -3.0608


In [14]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
rng = np.random.default_rng(10)

# --- 多重比较: 20 个基准上的 20 次比较, 其中 5 个真有差异 ---
m, n_eff = 20, 5
effects = np.r_[np.full(n_eff, 0.08), np.zeros(m - n_eff)]   # 真实效应
est = effects + rng.normal(0, 0.03, m)                        # 估计值
se = np.full(m, 0.03)
pvals = 2 * (1 - stats.norm.cdf(np.abs(est) / se))

for method, ctrl in [("bonferroni", "FWER"), ("holm", "FWER"),
                     ("fdr_bh", "FDR"), ("fdr_by", "FDR(任意相关)")]:
    rej, p_adj, _, _ = multipletests(pvals, alpha=0.05, method=method)
    fp = int((rej[n_eff:]).sum())                             # 假阳性数
    tp = int((rej[:n_eff]).sum())
    print(f"{method:11s} ({ctrl:12s}): 拒绝 {rej.sum():2d} 个  "
          f"真阳性={tp}/{n_eff}  假阳性={fp}")

# --- 功效分析: 由 pilot 方差求所需样本量与 MDE ---
sigma_d = 0.45                       # pilot 得到的配对差值标准差
alpha, power = 0.05, 0.80
z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
for delta in [0.05, 0.10, 0.20]:
    print(f"MDE={delta:.2f} -> n = {int(np.ceil(z**2 * sigma_d**2 / delta**2))}")
for n in [50, 100, 300, 1000]:
    print(f"n={n:4d} -> 80% 功效下可检出的 MDE = "
          f"{z * sigma_d / np.sqrt(n):.3f}")

# --- 评分者间一致性 ---
from sklearn.metrics import cohen_kappa_score
r1 = rng.integers(0, 2, 400); r2 = np.where(rng.random(400) < 0.15, 1 - r1, r1)
kap = cohen_kappa_score(r1, r2)
po = (r1 == r2).mean()
# kappa 的区间用 bootstrap 估计(解析式依赖边缘分布假设)
boot_k = [cohen_kappa_score(r1[i], r2[i])
          for i in (rng.integers(0, len(r1), (2000, len(r1))))
          if len(np.unique(r1[i])) > 1]
print(f"Cohen's kappa = {kap:.3f}  95% CI = {np.round(np.percentile(boot_k, [2.5, 97.5]), 3)}")
print(f"原始一致率 = {po:.3f}   边缘分布 = {np.round([r1.mean(), r2.mean()], 3)}")
print("注: kappa 依赖边缘分布(类别不平衡时 kappa 偏低), 报告时应同时给出一致率与边缘分布")

bonferroni  (FWER        ): 拒绝  0 个  真阳性=0/5  假阳性=0
holm        (FWER        ): 拒绝  0 个  真阳性=0/5  假阳性=0
fdr_bh      (FDR         ): 拒绝  0 个  真阳性=0/5  假阳性=0
fdr_by      (FDR(任意相关)   ): 拒绝  0 个  真阳性=0/5  假阳性=0
MDE=0.05 -> n = 636
MDE=0.10 -> n = 159
MDE=0.20 -> n = 40
n=  50 -> 80% 功效下可检出的 MDE = 0.178
n= 100 -> 80% 功效下可检出的 MDE = 0.126
n= 300 -> 80% 功效下可检出的 MDE = 0.073
n=1000 -> 80% 功效下可检出的 MDE = 0.040
Cohen's kappa = 0.665  95% CI = [0.592 0.739]
原始一致率 = 0.833   边缘分布 = [0.46  0.498]
注: kappa 依赖边缘分布(类别不平衡时 kappa 偏低), 报告时应同时给出一致率与边缘分布


In [ ]:
# import json, glob
# import numpy as np
# from scipy import stats

# def load_items(pattern, task="mmlu"):
#     """读取 harness 输出的 per-item 明细; 不同版本字段名可能不同"""
#     rows = []
#     for f in sorted(glob.glob(pattern)):
#         with open(f, encoding="utf-8") as fh:
#             data = json.load(fh)
#         res = data["results"].get(task, {})
#         rows.append(np.asarray(res.get("acc_per_item", []), dtype=float))
#     return np.vstack(rows)                       # (n_seed, n_item)

# A = load_items("./results/qwen25-7b-seed*.json")
# B = load_items("./results/llama31-8b-seed*.json")
# assert A.shape == B.shape, "两个模型必须使用同一批题目与同样的重复次数"

# a = A.mean(axis=0) > 0.5                        # 题目层: 多次重复的多数结果
# b = B.mean(axis=0) > 0.5
# b_disc = int(((a == 1) & (b == 0)).sum()); c_disc = int(((a == 0) & (b == 1)).sum())
# p_exact = stats.binomtest(b_disc, b_disc + c_disc, 0.5).pvalue
# print(f"McNemar 精确检验: b={b_disc} c={c_disc} p={p_exact:.4f}")

# rng = np.random.default_rng(0)
# n_item = A.shape[1]
# boot = [(b - a)[rng.integers(0, n_item, n_item)].mean() for _ in range(5000)]
# print(f"配对差值 = {(b.astype(float) - a.astype(float)).mean():+.4f}  "
#       f"95% CI = {np.round(np.percentile(boot, [2.5, 97.5]), 4)}")
# print(f"种子间标准差 A={A.mean(0).std():.4f} B={B.mean(0).std():.4f} "
#       f"(用于方差分量分解, 见 16.5.2)")